# TTBI Ablation Study

**What this notebook does:**

1. Defines the dataset, active DOFs, and discretisation step.
2. Builds the architecture ablation grid automatically.
3. Runs four sequential phases — isolated sensor sweep, architecture
   ablation, DOF sensitivity, and forward selection — each resumable
   from wherever it left off.

**What this notebook does NOT do:**

All model classes, training loops, plotting functions, and data-loading
logic live in importable modules.  Nothing here should need editing
when those details change.

## Imports

In [ ]:
import copy
import itertools

import optuna
from core.utils    import (set_global_seed, define_save_locations,
                            DOF_NAMES, DOF_NAME_TO_IDX, IDX_TO_DOF_NAME)
from training.pipeline      import execute_ablation_pipeline
from training.robustness    import evaluate_parametric_robustness
from plotting.robustness_plots import plot_stochastic_summary, plot_parametric_summary

optuna.logging.set_verbosity(optuna.logging.WARNING)

## Configuration

**Edit only this cell between runs.**

`DOFs` — which physical channels to include.  All eight are active for
the full ablation; a subset can be passed here to test a specific sensor
combination without touching any other cell.

`discretization` — damage class granularity in percent.

1  →  61 classes (0 %, 1 %, …, 60 %)

5  →  13 classes (0 %, 5 %, …, 60 %)

In [ ]:
DATASET        = "data_noise_vehicle_temperature"
DOFs           = list(range(8))     # all eight sensors
DISCRETIZATION = 1
GLOBAL_SEED    = 42

set_global_seed(GLOBAL_SEED)

# Confirm the active DOF names for the run log
print("Active DOFs:")
for i, dof in enumerate(DOFs):
    print(f"  {dof}  {IDX_TO_DOF_NAME[dof]}")

## Build the architecture ablation grid

16 one-dimensional models (RAW × 8 flag combinations + PAA × 8 flag
combinations) plus one two-dimensional CWT model = 17 configurations.

The grid is built programmatically so adding a new preprocessor or a new
architecture flag is a one-line change.

In [ ]:
ablation_grid  = []
counter        = 1
preprocessors  = ['RAW', 'PAA']
flag_values    = [True, False]
 
for prep in preprocessors:
    for use_s2v, use_lstm, use_nhits in itertools.product(flag_values, repeat=3):
 
        features   = []
        if use_s2v:  features.append("S2V")
        if use_lstm: features.append("LSTM")
        if use_nhits:features.append("NHiTS")
        feature_str = "_".join(features) if features else "Base"
 
        ablation_grid.append({
            "name":          f"{counter}_{prep}_{feature_str}",
            "method":        prep,
            "dofs":          DOFs,
            "discretization": DISCRETIZATION,
            "use_space2vec": use_s2v,
            "use_lstm":      use_lstm,
            "use_nhits":     use_nhits,
            "model_type":    "1D_MODULAR",
        })
        counter += 1
 
# CWT 2-D model
ablation_grid.append({
    "name":          f"{counter}_CWT_2D_CNN",
    "method":        "PAA_CWT",
    "dofs":          DOFs,
    "discretization": DISCRETIZATION,
    "use_space2vec": False,
    "use_lstm":      False,
    "use_nhits":     False,
    "model_type":    "2D_CNN",
})
 
print(f"{len(ablation_grid)} architectures in the ablation grid.")

## Phase 1 — Isolated single-sensor sweep

Trains all 17 architectures independently on each individual sensor
so every sensor's marginal utility can be measured without the
confound of other channels.

Results are stored per-sensor; KeyboardInterrupt saves all progress.

In [ ]:
isolated_results = {}

try:
    for dof in DOFs:
        sensor_name = IDX_TO_DOF_NAME[dof]
        print(f"\n{'*' * 52}")
        print(f"  Isolated sweep: {sensor_name}  (DOF {dof})")
        print(f"{'*' * 52}")
 
        grid = copy.deepcopy(ablation_grid)
        for cfg in grid:
            cfg["dofs"] = [dof]
 
        db, out, cache = define_save_locations(
            f"IsolatedSweep_{sensor_name}",
            dataset=DATASET,
            DOFs=[dof],
            discretization=DISCRETIZATION,
        )
 
        results = execute_ablation_pipeline(
            experiment_path=grid,
            database_name=db,
            output_dir_name=out,
            cache_dir_name=cache,
            dataset=DATASET,
            n_trials=100,
            epochs=50,
        )
        isolated_results[sensor_name] = results
 
        plot_stochastic_summary(
            path_list=grid,
            db_storage=db,
            experiment_root_dir=out,
            summary_output_dir=f"{out}/Summary_Plots",
            file_prefix=f"isolated_{sensor_name}",
        )
 
        print(f"  Isolated sweep complete: {sensor_name}")
 
except KeyboardInterrupt:
    completed = list(isolated_results.keys())
    print(f"\nInterrupted after {len(completed)} sensor(s): {completed}")
 
print("\nPhase 1 complete.")

## Phase 2 — Architecture ablation

Trains all 17 architectures on the full DOF set to find the best
architecture independent of sensor selection.

In [ ]:
db_arch, out_arch, cache_arch = define_save_locations(
    'architectures',
    dataset=DATASET,
    DOFs=DOFs,
    discretization=DISCRETIZATION,
)
 
architecture_results = execute_ablation_pipeline(
    experiment_path=ablation_grid,
    database_name=db_arch,
    output_dir_name=out_arch,
    cache_dir_name=cache_arch,
    dataset=DATASET,
    n_trials=50,
    epochs=50,
)
 
plot_stochastic_summary(
    path_list=ablation_grid,
    db_storage=db_arch,
    experiment_root_dir=out_arch,
    summary_output_dir=f"{out_arch}/Summary_Plots",
    file_prefix="architecture_ablation",
)

## Phase 2b — Champion selection and parametric stress test

The champion is the architecture with the lowest 95 % UCB MSE across the
30-seed Monte Carlo test — the most *reliably* good model, not just the
luckiest Optuna trial.

In [ ]:
# Select champion
champion_result = min(architecture_results, key=lambda x: x['UCB_95_MSE'])
champion_name   = champion_result['Model']
print(f"\nChampion: {champion_name}")
print(f"  Optuna score:       {champion_result['Optuna_Lucky_Score']:.4f}")
print(f"  Stochastic mean MSE:{champion_result['Stochastic_Mean_MSE']:.4f}")
print(f"  UCB 95 %:           {champion_result['UCB_95_MSE']:.4f}")
 
# Retrieve the champion's config and study
champion_config = next(c for c in ablation_grid if c['name'] == champion_name)
champion_study  = optuna.load_study(study_name=champion_name, storage=db_arch)
champion_dir    = f"{out_arch}/{champion_name}"

In [ ]:
# Parametric stress test on the champion only
worst_mse, max_degradation = evaluate_parametric_robustness(
    study=champion_study,
    config=champion_config,
    dataset_name=DATASET,
    baseline_mse=champion_result['Stochastic_Mean_MSE'],
    n_epochs=50,
    cache_dir=cache_arch,
    output_dir=champion_dir,
)
 
plot_parametric_summary(
    champion_name=champion_name,
    experiment_root_dir=out_arch,
    summary_output_dir=f"{out_arch}/Summary_Plots",
)
 
print(f"\nParametric worst MSE:  {worst_mse:.4f}")
print(f"Max degradation:       +{max_degradation:.4f}")

## Phase 3 — Leave-one-out DOF sensitivity

Fixes the champion architecture and drops one sensor at a time.
The sensor whose absence causes the largest UCB increase is the
most important.

In [ ]:
dofs_waterfall = []
 
# Baseline: all sensors, champion architecture
dofs_waterfall.append({
    **{k: champion_config[k]
       for k in ('method', 'use_lstm', 'use_nhits', 'use_space2vec', 'model_type')},
    "name":          f"Baseline_All_{len(DOFs)}_{champion_name}",
    "dofs":          DOFs.copy(),
    "discretization": DISCRETIZATION,
})
 
# Drop each sensor individually
for i, dropped_dof in enumerate(DOFs):
    remaining = [d for d in DOFs if d != dropped_dof]
    dofs_waterfall.append({
        **{k: champion_config[k]
           for k in ('method', 'use_lstm', 'use_nhits', 'use_space2vec', 'model_type')},
        "name":          f"Drop_{IDX_TO_DOF_NAME[dropped_dof]}",
        "dofs":          remaining,
        "discretization": DISCRETIZATION,
    })
 
print(f"{len(dofs_waterfall)} configurations in the DOF waterfall:")
for cfg in dofs_waterfall:
    print(f"  {cfg['name']}  ({len(cfg['dofs'])} DOFs)")
 
# %%
db_dofs, out_dofs, cache_dofs = define_save_locations(
    'DOFs_sensitivity',
    dataset=DATASET,
    DOFs=DOFs,
    discretization=DISCRETIZATION,
)
 
dofs_results = execute_ablation_pipeline(
    experiment_path=dofs_waterfall,
    database_name=db_dofs,
    output_dir_name=out_dofs,
    cache_dir_name=cache_dofs,
    dataset=DATASET,
    n_trials=50,
    epochs=50,
    skip_robustness=False,      # every model must complete the 30-seed test
)
 
plot_stochastic_summary(
    path_list=dofs_waterfall,
    db_storage=db_dofs,
    experiment_root_dir=out_dofs,
    summary_output_dir=f"{out_dofs}/Summary_Plots",
    file_prefix="sensor_elimination",
)

## Phase 3b — Sensor importance ranking

Models are sorted by the UCB MSE they produced when that sensor was
*removed*.  The largest increase in UCB = the most critical sensor.

In [ ]:
drop_results = [r for r in dofs_results if r['Model'].startswith("Drop_")]
drop_results.sort(key=lambda x: x['UCB_95_MSE'], reverse=True)
 
ranked_dofs = []
print("Sensor importance ranking (most → least critical):")
for rank, res in enumerate(drop_results, start=1):
    sensor_name = res['Model'].replace("Drop_", "")
    dof_idx     = DOF_NAME_TO_IDX[sensor_name]
    ranked_dofs.append(dof_idx)
    print(f"  {rank}. {sensor_name}  "
          f"(UCB without it: {res['UCB_95_MSE']:.4f})")

## Phase 4 — Cross-architecture forward selection sweep

Adds sensors one at a time in importance order and trains all 17
architectures at each sensor count.  Finds the minimum sensor set
that recovers the full-set UCB.

In [ ]:
cumulative_dofs      = []
forward_sweep_results = {}
 
try:
    for i, next_dof in enumerate(ranked_dofs):
        cumulative_dofs.append(next_dof)
        n_sensors    = len(cumulative_dofs)
        phase_name   = f"ForwardSweep_{n_sensors}Sensors"
 
        print(f"\n{'*' * 52}")
        print(f"  {phase_name}  —  DOFs: {cumulative_dofs}")
        print(f"{'*' * 52}")
 
        grid = copy.deepcopy(ablation_grid)
        for cfg in grid:
            cfg["dofs"] = cumulative_dofs.copy()
 
        db_sw, out_sw, cache_sw = define_save_locations(
            phase_name,
            dataset=DATASET,
            DOFs=cumulative_dofs,
            discretization=DISCRETIZATION,
        )
 
        sweep_results = execute_ablation_pipeline(
            experiment_path=grid,
            database_name=db_sw,
            output_dir_name=out_sw,
            cache_dir_name=cache_sw,
            dataset=DATASET,
            n_trials=50,
            epochs=50,
        )
        forward_sweep_results[n_sensors] = sweep_results
 
        plot_stochastic_summary(
            path_list=grid,
            db_storage=db_sw,
            experiment_root_dir=out_sw,
            summary_output_dir=f"{out_sw}/Summary_Plots",
            file_prefix=f"forward_{n_sensors}_sensors",
        )
 
        print(f"  Forward sweep complete: {n_sensors} sensor(s).")
 
except KeyboardInterrupt:
    print(f"\nInterrupted after {len(cumulative_dofs) - 1} sensor(s).")
    print("All data up to this point is saved.")
 
print("\nEntire ablation pipeline complete.")